# TP3 Recomendador personalizado de libros

## Paso 1 Cargar y revisar los libros

In [3]:
import pandas as pd
import os
file_path = '/catalogo_tp3.jsonl'
df = pd.read_json(file_path, lines=True)


In [4]:
# Vemos información general sobre las columnas y tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   object             
 1   slug            8747 non-null   object             
 2   title           8747 non-null   object             
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   object             
 6   series          2316 non-null   object             
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   object             
 9   synopsis        8746 non-null   object             
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(1), int64(1), object(9)
memory usage: 820.2+ KB


In [5]:
# Conservamos solo las columnas requeridas
columnas_a_mantener = ['slug', 'title', 'authors', 'synopsis']
df_filtrado = df[columnas_a_mantener].copy()

# ponemos como id la columna slug? o solo identificamos el libro con ese titulo

# Eliminamos los registros sin sinopsis
df_filtrado = df_filtrado.dropna(subset=['synopsis'])
df_filtrado = df_filtrado[df_filtrado['synopsis'].str.strip() != '']

# Eliminamos duplicados por el identificador 'slug'
df_filtrado = df_filtrado.drop_duplicates(subset=['slug'])

# Mostramos la cantidad de registros restantes y una vista previa
print(f"Cantidad de registros restantes: {len(df_filtrado)}")
display(df_filtrado.head())

Cantidad de registros restantes: 8746


,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano\n...


In [6]:
import re

# Normalizamos espacios en blanco
def normalizar_espacios(texto):
    if isinstance(texto, str):
        # \s+ coincide con cualquier espacio en blanco, incluidos \n, \r, \t, y reemplaza por uno unico
        return re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicamos la funcion para normalizar las columnas de texto relevantes
columnas_texto = ['title', 'authors', 'synopsis']
for col in columnas_texto:
    df_filtrado[col] = df_filtrado[col].apply(normalizar_espacios)

# Mostramos para verificar los resultados
display(df_filtrado.head())

,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...


In [7]:
# Contar la cantidad de registros antes de eliminar duplicados de sinopsis
total_antes = len(df_filtrado)

# Identificar y eliminar duplicados basados únicamente en la columna 'synopsis'
# 'keep="first"' conserva el primer libro que aparece con esa sinopsis
df_filtrado = df_filtrado.drop_duplicates(subset=['synopsis'], keep='first')

# Calcular cuántos registros se eliminaron
total_despues = len(df_filtrado)
eliminados = total_antes - total_despues

print(f"Cantidad de registros antes: {total_antes}")
print(f"Cantidad de registros después de eliminar sinopsis duplicadas: {total_despues}")
print(f"Se eliminaron {eliminados} libros con sinopsis repetidas.")

Cantidad de registros antes: 8746
Cantidad de registros después de eliminar sinopsis duplicadas: 8648
Se eliminaron 98 libros con sinopsis repetidas.


In [8]:
!pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [thefuzz]


In [9]:
from thefuzz import fuzz
import pandas as pd

# Convertiremos los autores a string para agrupar de forma limpia
df_filtrado['authors_str'] = df_filtrado['authors'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

# Listas para guardar los ejemplos de unir y separar
ejemplos_unidos = []
ejemplos_separados = []

# Conjunto para rastrear los índices de registros a eliminar
indices_a_eliminar = set()

# Agrupamos por autor para evitar la comparación de todos contra todos
grupos = df_filtrado.groupby('authors_str')

for autor, grupo in grupos:
    if len(grupo) < 2:
        continue

    # Convertimos el grupo a una lista de diccionarios para iterar fácilmente
    libros = grupo.to_dict('records')
    n = len(libros)

    for i in range(n):
        idx_i = grupo.index[i]
        if idx_i in indices_a_eliminar:
            continue

        for j in range(i + 1, n):
            idx_j = grupo.index[j]
            if idx_j in indices_a_eliminar:
                continue

            titulo_i = libros[i]['title']
            titulo_j = libros[j]['title']
            sinopsis_i = libros[i]['synopsis']
            sinopsis_j = libros[j]['synopsis']

            # Calculamos similitudes
            sim_titulo = fuzz.token_sort_ratio(titulo_i.lower(), titulo_j.lower())
            sim_sinopsis = fuzz.token_sort_ratio(sinopsis_i.lower(), sinopsis_j.lower())

            # Criterio:
            # 1. Si los títulos son casi idénticos (ej. > 95%)
            # 2. Si los títulos son muy similares (ej. > 80%) Y la sinopsis tiene similitud moderada (ej. > 50%)
            deben_unirse = (sim_titulo > 95) or (sim_titulo > 80 and sim_sinopsis > 50)

            # Evitar unir obras numeradas de forma evidente (como Tomo I vs Tomo II)
            # Si el título contiene números romanos o dígitos diferentes al final, no los unimos
            es_secuela = False
            palabras_i = set(titulo_i.lower().split())
            palabras_j = set(titulo_j.lower().split())
            diferencia = palabras_i.symmetric_difference(palabras_j)
            secuela_keywords = {'i', 'ii', 'iii', 'iv', 'v', 'vi', '1', '2', '3', '4', '5'}
            if diferencia.intersection(secuela_keywords):
                es_secuela = True

            if deben_unirse and not es_secuela:
                indices_a_eliminar.add(idx_j)
                if len(ejemplos_unidos) < 2:
                    ejemplos_unidos.append((
                        (titulo_i, sinopsis_i[:150] + "..."),
                        (titulo_j, sinopsis_j[:150] + "..."),
                        f"Sim. Título: {sim_titulo}%, Sim. Sinopsis: {sim_sinopsis}%"
                    ))
            else:
                if len(ejemplos_separados) < 2 and (sim_titulo > 70):
                    ejemplos_separados.append((
                        (titulo_i, sinopsis_i[:150] + "..."),
                        (titulo_j, sinopsis_j[:150] + "..."),
                        f"Sim. Título: {sim_titulo}%, Sim. Sinopsis: {sim_sinopsis}%"
                    ))

# Creamos el dataframe final sin las obras consideradas duplicadas/versiones casi iguales
df_final = df_filtrado.drop(index=list(indices_a_eliminar))

# Imprimimos resultados
print(f"Registros antes de remover versiones casi iguales: {len(df_filtrado)}")
print(f"Registros actuales: {len(df_final)}")
print(f"Se removieron {len(indices_a_eliminar)} versiones duplicadas.\n")

Registros antes de remover versiones casi iguales: 8648
Registros actuales: 8575
Se removieron 73 versiones duplicadas.



In [10]:
print("=== DOS PARES QUE DECIDIMOS UNIR ===")
for idx, (obra1, obra2, razon) in enumerate(ejemplos_unidos, 1):
    print(f"\nPar {idx} ({razon}):")
    print(f"  - Obra A: \"{obra1[0]}\" | Sinopsis: {obra1[1]}")
    print(f"  - Obra B: \"{obra2[0]}\" | Sinopsis: {obra2[1]}")

print("\n=== DOS PARES QUE DECIDIMOS SEPARAR ===")
for idx, (obra1, obra2, razon) in enumerate(ejemplos_separados, 1):
    print(f"\nPar {idx} ({razon}):")
    print(f"  - Obra A: \"{obra1[0]}\" | Sinopsis: {obra1[1]}")
    print(f"  - Obra B: \"{obra2[0]}\" | Sinopsis: {obra2[1]}")

=== DOS PARES QUE DECIDIMOS UNIR ===

Par 1 (Sim. Título: 100%, Sim. Sinopsis: 55%):
  - Obra A: "Nuevos cuentos de Bustos Domecq" | Sinopsis: Caso ejemplar y a la vez rarísimo de escritura en colaboración, H. Bustos Domecq no es, sin embargo, la suma de sus dos ilustres creadores ni es asimi...
  - Obra B: "Nuevos cuentos de Bustos Domecq" | Sinopsis: Hace cincuenta años que H. Bustos Domecq ocupa un puesto de honor en el poco dilatado territorio de la literatura argentina. Nacido de la amistad entr...

Par 2 (Sim. Título: 91%, Sim. Sinopsis: 57%):
  - Obra A: "Historias negras del fútbol argentino" | Sinopsis: Hinchas que pretenden ahorcar a un árbitro, dirigentes que practican el soborno a repetición, maniobras para evitar descensos, sobre todo cuando afect...
  - Obra B: "Nuevas historias negras del fútbol argentino" | Sinopsis: Denuncias por sobornos, partidos con resultados mentirosos, otros consensuados a tal punto que el jugador que mete un gol, en vez de gritarlo, se agar...



In [11]:
# 1. Mostrar 5 registros de muestra del catálogo final limpio
print("=== 5 REGISTROS DE MUESTRA DEL CATÁLOGO LIMPIO ===")
display(df_final.head(5))

# 2. Calcular estadísticas de eliminación paso a paso
registros_iniciales = 8747

# Paso 1: Sinopsis vacías/nulas y duplicados de slug
# Inicialmente: 8747
# Después de dropna y slug deduplication: 8746 (se eliminaron 1)
sinopsis_vacias_o_duplicados_slug = registros_iniciales - 8746

# Paso 2: Sinopsis idénticas
# De 8746 a 8648 (se eliminaron 98)
eliminados_sinopsis_idencitas = 8746 - 8648

# Paso 3: Versiones casi iguales del mismo autor
# De 8648 a 8575 (se eliminaron 73)
eliminados_versiones_casi_iguales = 8648 - 8575

total_disponibles = len(df_final)
total_eliminados = registros_iniciales - total_disponibles

# 3. Mostramos resumen del proceso
print("\n=== RESUMEN DEL PROCESO DE LIMPIEZA ===")
print(f"Libros disponibles al final: {total_disponibles}")
print(f"Total de libros eliminados: {total_eliminados}")
print(f"  - Por sinopsis vacías o identificador (slug) duplicado: {sinopsis_vacias_o_duplicados_slug}")
print(f"  - Por sinopsis exactamente idénticas (mismo curso/serie): {eliminados_sinopsis_idencitas}")
print(f"  - Por versiones casi iguales del mismo autor (mismo título/sinopsis similar): {eliminados_versiones_casi_iguales}")

=== 5 REGISTROS DE MUESTRA DEL CATÁLOGO LIMPIO ===


,slug,title,authors,synopsis,authors_str
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...,Eudald Carbonell
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...,David Reich
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...,Eduardo Matos Moctezuma
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...,M.ª Eugenia Aubet
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...,Emiliano Bruner



=== RESUMEN DEL PROCESO DE LIMPIEZA ===
Libros disponibles al final: 8575
Total de libros eliminados: 172
  - Por sinopsis vacías o identificador (slug) duplicado: 1
  - Por sinopsis exactamente idénticas (mismo curso/serie): 98
  - Por versiones casi iguales del mismo autor (mismo título/sinopsis similar): 73


Al terminar el paso 1 quedan **8575 libros** disponibles. Se eliminaron 172: 1 sin sinopsis, 98 por sinopsis exactamente repetidas (cursos y series que repiten el mismo texto) y 73 por ser versiones casi iguales de otra obra del mismo autor.